In [2]:
import pandas as pd
import duckdb
data = [
    # Device A：缺少 2026-07-03
    ["A", "2026-07-01", 10],
    ["A", "2026-07-02", 12],
    ["A", "2026-07-04", 15],
    ["A", "2026-07-05", 9],

    # Device B：日期连续
    ["B", "2026-07-01", 5],
    ["B", "2026-07-02", 8],
    ["B", "2026-07-03", 8],
    ["B", "2026-07-04", 6],

    # Device C：缺少 2026-07-02
    ["C", "2026-07-01", 20],
    ["C", "2026-07-03", 25],
    ["C", "2026-07-04", 22],
]

df = pd.DataFrame(
    data,
    columns=["device_id", "stat_date", "alarm_count"]
)

df["stat_date"] = pd.to_datetime(df["stat_date"])

print(df)

df.to_csv("device_daily_alarm_missing_dates_log.csv", index=False)

   device_id  stat_date  alarm_count
0          A 2026-07-01           10
1          A 2026-07-02           12
2          A 2026-07-04           15
3          A 2026-07-05            9
4          B 2026-07-01            5
5          B 2026-07-02            8
6          B 2026-07-03            8
7          B 2026-07-04            6
8          C 2026-07-01           20
9          C 2026-07-03           25
10         C 2026-07-04           22


## 题目要求

* **分别使用 SQL 和 Pandas 完成：**

- 计算每个设备每天的报警次数与前一天相比变化了多少。

**注意，这里说的是：**

前一天

**不是：**

上一条记录

**如果前一天没有记录，则：**

- `previous_day_alarm_count = NULL / NaN`
- `alarm_count_diff = NULL / NaN`

* **最终输出字段：**

- `device_id`
- `stat_date`
- `alarm_count`
- `previous_day_alarm_count`
- `alarm_count_diff`

In [6]:
# SQL轨道

query = """
WITH previous_table AS(
SELECT
    device_id,
    stat_date + INTERVAL 1day AS stat_date,
    alarm_count AS previous_alarm_count
FROM df
)
SELECT
    curr.device_id,
    curr.stat_date,
    curr.alarm_count,
    pre.previous_alarm_count,
    curr.alarm_count - pre.previous_alarm_count AS alarm_count_diff
FROM df AS curr
LEFT JOIN previous_table AS pre
ON curr.device_id = pre.device_id
    AND curr.stat_date = pre.stat_date
ORDER BY device_id,stat_date

"""
df_sql = duckdb.execute(query).fetchdf()
df_sql

,device_id,stat_date,alarm_count,previous_alarm_count,alarm_count_diff
0,A,2026-07-01,10,<NA>,<NA>
1,A,2026-07-02,12,10,2
2,A,2026-07-04,15,<NA>,<NA>
3,A,2026-07-05,9,15,-6
4,B,2026-07-01,5,<NA>,<NA>
5,B,2026-07-02,8,5,3
6,B,2026-07-03,8,8,0
7,B,2026-07-04,6,8,-2
8,C,2026-07-01,20,<NA>,<NA>
9,C,2026-07-03,25,<NA>,<NA>


In [12]:
# PANDAS轨道

df_prev = (
    df
    .assign(
        stat_date = lambda x:x['stat_date'] + pd.Timedelta(days=1)
    )
    .rename(
        columns = {'alarm_count':'previous_alarm_count'}
    )
)
df_pd = (
    df
    .merge(
        df_prev,
        on = ['device_id','stat_date'],
        how = 'left'
    )
    .assign(
        alarm_count_diff = lambda x:(
            x['alarm_count'] - x['previous_alarm_count']
    )
        )
    .sort_values(by=['device_id', 'stat_date'])
    .reset_index(drop=True)
)
df_pd

,device_id,stat_date,alarm_count,previous_alarm_count,alarm_count_diff
0,A,2026-07-01,10,NaN,NaN
1,A,2026-07-02,12,10.0,2.0
2,A,2026-07-04,15,NaN,NaN
3,A,2026-07-05,9,15.0,-6.0
4,B,2026-07-01,5,NaN,NaN
5,B,2026-07-02,8,5.0,3.0
6,B,2026-07-03,8,8.0,0.0
7,B,2026-07-04,6,8.0,-2.0
8,C,2026-07-01,20,NaN,NaN
9,C,2026-07-03,25,NaN,NaN
